# 01 · First Whisper fine-tune for Fon

Fine-tunes Whisper on the speaker-disjoint splits made by `prepare_data.py`.

1. Run `python prepare_data.py` once (and again after downloading more audio).
2. Pick `RUN = "smoke"` to check the pipeline on this Mac (whisper-tiny, 200 clips, a few minutes),
   or `RUN = "full"` on a GPU (whisper-small, all clips, ~4,000 steps).

In [1]:
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import jiwer
import numpy as np
import soundfile as sf
import torch
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

sys.path.insert(0, ".")
sys.path.insert(0, "../kanbior-mcore")
from fon_text import normalize_for_wer
from prepare_data import read_split

/Users/yetodo/Documents/meall/kanbior/.kanbior-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1122 18:41:25.247000 19741 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## Configuration

In [2]:
RUN = "smoke"  # "smoke" (Mac, pipeline check) or "full" (GPU, real training)

DATA_DIR = Path("sample-data")          # on Kaggle e.g. Path("/kaggle/input/jml-fon-sample")
SPLITS_DIR = DATA_DIR / "splits"
OUTPUT_DIR = Path("runs") / f"whisper-fon-{RUN}"

PRESETS = {
    "smoke": dict(model="openai/whisper-tiny", max_train=200, max_eval=40, max_steps=100,
                  batch_size=8, lr=3.75e-5, warmup=10, eval_every=50),
    "full": dict(model="openai/whisper-small", max_train=None, max_eval=None, max_steps=4000,
                 batch_size=16, lr=1e-5, warmup=500, eval_every=500),
}
cfg = PRESETS[RUN]

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Run:", RUN, "| device:", device, "|", cfg)

Run: smoke | device: mps | {'model': 'openai/whisper-tiny', 'max_train': 200, 'max_eval': 40, 'max_steps': 100, 'batch_size': 8, 'lr': 3.75e-05, 'warmup': 10, 'eval_every': 50}


## Data

In [3]:
def take(rows, n, seed=0):
    """Random subset of n rows (all rows if n is None), reproducible."""
    if n is None or n >= len(rows):
        return rows
    idx = np.random.default_rng(seed).choice(len(rows), size=n, replace=False)
    return [rows[i] for i in sorted(idx)]

train_rows = take(read_split(SPLITS_DIR / "train.csv"), cfg["max_train"])
val_rows = take(read_split(SPLITS_DIR / "val.csv"), cfg["max_eval"])
test_rows = take(read_split(SPLITS_DIR / "test.csv"), cfg["max_eval"])
for name, rows in [("train", train_rows), ("val", val_rows), ("test", test_rows)]:
    hours = sum(r["duration"] for r in rows) / 3600
    print(f"{name:5s}: {len(rows):5,} clips, {hours:5.2f} h, {len({r['speaker_id'] for r in rows})} speakers")

train:   200 clips,  0.41 h, 60 speakers
val  :    40 clips,  0.09 h, 25 speakers
test :    40 clips,  0.08 h, 21 speakers


## Processor and tokenizer check

Whisper has no Fon language token, so we reuse French (as finanfa-fon-asr-v6 does).

In [4]:
processor = WhisperProcessor.from_pretrained(cfg["model"], language="french", task="transcribe")
tokenizer = processor.tokenizer

all_texts = [r["text"] for r in read_split(SPLITS_DIR / "train.csv")]
lengths = [len(tokenizer(t).input_ids) for t in all_texts]
bad_roundtrip = [t for t in all_texts if tokenizer.decode(tokenizer(t).input_ids, skip_special_tokens=True) != t]
print(f"Label tokens: mean {np.mean(lengths):.0f}, max {max(lengths)} (Whisper limit 448)")
print(f"Texts that don't survive encode/decode: {len(bad_roundtrip)} of {len(all_texts)}")
print("Example:", all_texts[0], "->", len(tokenizer(all_texts[0]).input_ids), "tokens")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Label tokens: mean 64, max 197 (Whisper limit 448)
Texts that don't survive encode/decode: 0 of 6550
Example: Kèmi ná yi azɔ̌mɛ ɖo Canada é ná yí ɖo semaine e jǎwe ɔ́ mɛ. -> 48 tokens


In [5]:
class FonSpeechDataset(torch.utils.data.Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        audio, sr = sf.read(DATA_DIR / row["audio_path"], dtype="float32")
        assert sr == 16000, f"{row['id']}: expected 16 kHz, got {sr}"
        return {"audio": audio, "text": row["text"]}


@dataclass
class Collator:
    processor: WhisperProcessor

    def __call__(self, batch):
        features = self.processor.feature_extractor(
            [b["audio"] for b in batch], sampling_rate=16000, return_tensors="pt"
        )
        labels = self.processor.tokenizer([b["text"] for b in batch], padding=True, return_tensors="pt")
        label_ids = labels.input_ids.masked_fill(labels.attention_mask.ne(1), -100)
        # The model prepends <|startoftranscript|> itself; drop it if the tokenizer added it.
        if (label_ids[:, 0] == model.config.decoder_start_token_id).all():
            label_ids = label_ids[:, 1:]
        return {"input_features": features.input_features, "labels": label_ids}


train_ds, val_ds, test_ds = (FonSpeechDataset(r) for r in (train_rows, val_rows, test_rows))

## Model

In [6]:
model = WhisperForConditionalGeneration.from_pretrained(cfg["model"])
model.generation_config.language = "fr"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
print(f"{cfg['model']}: {model.num_parameters() / 1e6:.0f}M parameters")

Loading weights: 100%|██████████| 167/167 [00:00<00:00, 10452.59it/s]

openai/whisper-tiny: 38M parameters


## Metrics

WER and CER on normalized text (lowercase, no punctuation), with and without tone marks.

In [8]:
def score(refs, preds):
    out = {}
    for suffix, keep_tones in (("", True), ("_no_tones", False)):
        r = [normalize_for_wer(t, keep_tones) for t in refs]
        p = [normalize_for_wer(t, keep_tones) for t in preds]
        out["wer" + suffix] = 100 * jiwer.wer(r, p)
        out["cer" + suffix] = 100 * jiwer.cer(r, p)
    return out


def compute_metrics(eval_pred):
    pred_ids, label_ids = eval_pred.predictions, eval_pred.label_ids
    pred_ids = np.where(pred_ids < 0, tokenizer.pad_token_id, pred_ids)
    label_ids = np.where(label_ids < 0, tokenizer.pad_token_id, label_ids)
    preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    refs = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return score(refs, preds)

## Training

In [9]:
args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=cfg["batch_size"],
    per_device_eval_batch_size=cfg["batch_size"],
    learning_rate=cfg["lr"],
    warmup_steps=cfg["warmup"],
    max_steps=cfg["max_steps"],
    gradient_checkpointing=(device == "cuda"),
    fp16=(device == "cuda"),
    eval_strategy="steps",
    eval_steps=cfg["eval_every"],
    save_steps=cfg["eval_every"],
    save_total_limit=2,
    logging_steps=10,
    predict_with_generate=True,
    generation_max_length=225,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    remove_unused_columns=False,
    dataloader_num_workers=0 if device != "cuda" else 2,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=Collator(processor),
    compute_metrics=compute_metrics,
    processing_class=processor,
)

/Users/yetodo/Documents/meall/kanbior/.kanbior-env/lib/python3.12/site-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)


In [10]:
# Baseline: the untrained model on the validation set.
baseline = trainer.evaluate()
print({k: round(v, 1) for k, v in baseline.items() if "er" in k})

/Users/yetodo/Documents/meall/kanbior/.kanbior-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Training Loss,Validation Loss,Step,Wer,Cer,Wer No Tones,Cer No Tones
No log,4.291602,0,170.861833,136.319534,170.725034,136.223441


{'eval_wer': 170.9, 'eval_cer': 136.3, 'eval_wer_no_tones': 170.7, 'eval_cer_no_tones': 136.2}


In [11]:
start = time.time()
trainer.train()
print(f"Training took {(time.time() - start) / 60:.1f} min")

Step,Training Loss,Validation Loss,Wer,Cer,Wer No Tones,Cer No Tones
50,1.006000,1.209717,106.839945,60.672704,97.948016,50.582591
100,0.495568,1.102782,108.891929,52.199224,100.410397,45.579164


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.28it/s]
/Users/yetodo/Documents/meall/kanbior/.kanbior-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.27it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Training took 2.8 min


In [12]:
# Training loss and validation WER over time.
for log in trainer.state.log_history:
    if "loss" in log:
        print(f"step {log['step']:5d}  loss {log['loss']:.3f}")
    if "eval_wer" in log:
        print(f"step {log['step']:5d}  val WER {log['eval_wer']:.1f}%  (no tones {log['eval_wer_no_tones']:.1f}%)")

step    10  loss 3.629
step    20  loss 2.164
step    30  loss 1.465
step    40  loss 1.122
step    50  loss 1.006
step    50  val WER 106.8%  (no tones 97.9%)
step    60  loss 0.706
step    70  loss 0.664
step    80  loss 0.551
step    90  loss 0.498
step   100  loss 0.496
step   100  val WER 108.9%  (no tones 100.4%)


## Test set

Speakers here were never seen in training or used to pick the best checkpoint.

In [13]:
test_metrics = trainer.evaluate(test_ds, metric_key_prefix="test")
print({k: round(v, 1) for k, v in test_metrics.items() if "er" in k})

/Users/yetodo/Documents/meall/kanbior/.kanbior-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step,Wer,Cer,Wer No Tones,Cer No Tones
0.495568,1.339162,100,88.994565,52.917903,81.657609,41.021188


{'test_wer': 89.0, 'test_cer': 52.9, 'test_wer_no_tones': 81.7, 'test_cer_no_tones': 41.0}


In [14]:
from IPython.display import Audio, display

def transcribe_rows(rows):
    model.eval()
    for row in rows:
        audio, _ = sf.read(DATA_DIR / row["audio_path"], dtype="float32")
        feats = processor.feature_extractor(audio, sampling_rate=16000, return_tensors="pt").input_features
        with torch.no_grad():
            ids = model.generate(feats.to(model.device, model.dtype), max_length=225)
        pred = tokenizer.decode(ids[0], skip_special_tokens=True).strip()
        display(Audio(str(DATA_DIR / row["audio_path"])))
        print(f"{row['id']} ({row['speaker_id']})")
        print("Reference:", row["text"])
        print("Predicted:", pred, "\n")

transcribe_rows(test_rows[:3])

tr_000021 (JML_S303)
Reference: Kokló nɔ kwásí zǎnzǎn tɛgbɛ hwɛcó bɔ hwesivɔ nɔ nyí tíntɔ́n tɔ́.
Predicted: Cloklɔ́ nɔ́ kpɛ́sí zɔnɔn tɛ́gbɛn bɔ jó bɔ́ kpɔ síwɔ nɔ́ yí tín tɔ́ ɖɔn. 



tr_000103 (JML_S095)
Reference: E ló a? Nɛ̌ a ka bló gbɔn?
Predicted: Éloa, l'éagga blokun. 



tr_000132 (JML_S514)
Reference: Mǐ mɔ nɔtɛn ɖokpó, nɔtɛn élɔ́ é nyɔ́ kpɔ́n tawun bɔ é jɛ́ xá ɖɔ xá mǐtɔn mɛ ɖésú ɔ́ è nǎ wa bló xɔ mɔ̌hunkɔtɔn, é nyɔ́ ɖɛkpɛ tawun bɔ gbɛtɔ́ lɛ́ɛ ɖ'é mɛ.
Predicted: Nǐ mɛ nɔ tɛn ɖɛ́n ɖě xɛ́n ɖě xɛ́n wɔ́ ká mɛ tɔn mɛ́ ɖi sú wɛ ná bló xɔn tɛ́n kɔ́ tɔ́ ɖu é nyɛ́nɛ kɛn tɔn bɛ́ɛ́ɛ́ ɖě hɛ́n. 



## Save

In [15]:
final_dir = OUTPUT_DIR / "final"
trainer.save_model(str(final_dir))
processor.save_pretrained(str(final_dir))
print("Saved to", final_dir)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.01it/s]


Saved to runs/whisper-fon-smoke/final
